# 💧 Delhi Deep Dive — Module 05: Water Scarcity Risk
## Groundwater depletion scoring for 11 Delhi districts (2010–2023)

In [ ]:
import os, time, warnings, pickle
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
warnings.filterwarnings('ignore')

OUTPUT_DIR = '../data/outputs/delhi'
os.makedirs(OUTPUT_DIR, exist_ok=True)
START, END = '2010-01-01', '2023-12-31'

DELHI_DISTRICTS = {
    'Central Delhi':    {'lat': 28.6508, 'lon': 77.2219},
    'East Delhi':       {'lat': 28.6271, 'lon': 77.2907},
    'New Delhi':        {'lat': 28.6139, 'lon': 77.2090},
    'North Delhi':      {'lat': 28.7041, 'lon': 77.1025},
    'North East Delhi': {'lat': 28.6921, 'lon': 77.2979},
    'North West Delhi': {'lat': 28.7219, 'lon': 77.0878},
    'Shahdara':         {'lat': 28.6735, 'lon': 77.2894},
    'South Delhi':      {'lat': 28.5244, 'lon': 77.1855},
    'South East Delhi': {'lat': 28.5677, 'lon': 77.2965},
    'South West Delhi': {'lat': 28.5672, 'lon': 77.0699},
    'West Delhi':       {'lat': 28.6541, 'lon': 77.0832},
}

# CGWB-based groundwater stress index (1.0 = critically over-exploited)
GROUNDWATER_STRESS = {
    'South Delhi': 1.0, 'South West Delhi': 0.95, 'West Delhi': 0.90,
    'North West Delhi': 0.85, 'Central Delhi': 0.80, 'New Delhi': 0.75,
    'South East Delhi': 0.85, 'North Delhi': 0.60, 'East Delhi': 0.55,
    'Shahdara': 0.50, 'North East Delhi': 0.45,
}

def score_to_category(s):
    if s <= 25: return 'LOW'
    elif s <= 50: return 'MEDIUM'
    elif s <= 75: return 'HIGH'
    return 'VERY HIGH'

def fetch_with_retry(url, params, max_retries=5):
    for attempt in range(max_retries):
        r = requests.get(url, params=params, timeout=30)
        if r.status_code == 429:
            wait = 5 * (2 ** attempt)
            print(f'⏳ {wait}s…', end=' ', flush=True)
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError('Max retries')

print('Water Scarcity module setup complete.')

## Fetch Rainfall + ET₀ Data

In [ ]:
dfs = []
for district, coords in DELHI_DISTRICTS.items():
    print(f'  {district}…', end=' ', flush=True)
    params = {
        'latitude': coords['lat'], 'longitude': coords['lon'],
        'start_date': START, 'end_date': END,
        'daily': ['precipitation_sum', 'et0_fao_evapotranspiration'],
        'timezone': 'Asia/Kolkata'
    }
    data = fetch_with_retry('https://archive-api.open-meteo.com/v1/archive', params)['daily']
    df = pd.DataFrame({
        'date':       pd.to_datetime(data['time']),
        'rainfall_mm': pd.to_numeric(data['precipitation_sum'], errors='coerce').fillna(0),
        'et0_mm':     pd.to_numeric(data['et0_fao_evapotranspiration'], errors='coerce').fillna(0),
    })
    df['district'] = district
    df['lat'] = coords['lat']
    df['lon'] = coords['lon']
    dfs.append(df)
    print('✓')
    time.sleep(1.5)

daily = pd.concat(dfs, ignore_index=True)
daily['year']  = daily['date'].dt.year
daily['month'] = daily['date'].dt.month
print(f'\nTotal: {len(daily):,} rows')

## Feature Engineering + Label + Train + Save

In [ ]:
def yearly_ws_features(g):
    g = g.sort_values('date')
    monsoon  = g[g['month'].isin([6,7,8,9])]
    premonso = g[g['month'].isin([3,4,5])]
    deficit  = float(max(g['et0_mm'].sum() - g['rainfall_mm'].sum(), 0))
    dry_days = int((g['rainfall_mm'] < 1).sum())
    return pd.Series({
        'water_deficit_mm':    deficit,
        'dry_days':            dry_days,
        'monsoon_rainfall_mm': float(monsoon['rainfall_mm'].sum()),
        'pre_monsoon_deficit': float(max(premonso['et0_mm'].sum() - premonso['rainfall_mm'].sum(), 0)),
    })

yearly = daily.groupby(['district','year']).apply(yearly_ws_features).reset_index()
for d, c in DELHI_DISTRICTS.items():
    yearly.loc[yearly['district']==d, 'lat'] = c['lat']
    yearly.loc[yearly['district']==d, 'lon'] = c['lon']

yearly['groundwater_stress_index'] = yearly['district'].map(GROUNDWATER_STRESS)
yearly = yearly.sort_values(['district','year'])
yearly['cumulative_deficit_3yr'] = yearly.groupby('district')['water_deficit_mm'].transform(
    lambda s: s.rolling(3, min_periods=1).sum())
yearly['waterscarcity_label'] = (
    (yearly['groundwater_stress_index'] > 0.7) &
    (yearly['water_deficit_mm'] > 1200)
).astype(int)

FEATURES = ['water_deficit_mm','dry_days','groundwater_stress_index',
            'cumulative_deficit_3yr','monsoon_rainfall_mm','pre_monsoon_deficit']
TARGET   = 'waterscarcity_label'

train = yearly[yearly['year']<=2021].dropna(subset=FEATURES+[TARGET])
test  = yearly[yearly['year']>=2022].dropna(subset=FEATURES+[TARGET])

model = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42)
model.fit(train[FEATURES].astype(float), train[TARGET].astype(int))
if len(test) > 0:
    print(f'Test acc: {accuracy_score(test[TARGET], model.predict(test[FEATURES].astype(float))):.3f}')

full  = yearly.dropna(subset=FEATURES+[TARGET]).copy()
probs = model.predict_proba(full[FEATURES].astype(float))[:,1]
full['waterscarcity_prob']       = probs.round(4)
full['waterscarcity_risk_score'] = (probs * 100).round(2)
full['risk_category']            = full['waterscarcity_risk_score'].apply(score_to_category)

# Feature importance
feat_df = pd.DataFrame({'feature': FEATURES, 'importance': model.feature_importances_}).sort_values('importance')
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(feat_df['feature'], feat_df['importance'], color='teal', alpha=0.85)
ax.set_title('Delhi Water Scarcity — Feature Importances')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/delhi_waterscarcity_feature_importance.png', dpi=150)
plt.show()

result = full[['district','year','lat','lon','waterscarcity_label','waterscarcity_prob','waterscarcity_risk_score','risk_category']]
result.to_csv(f'{OUTPUT_DIR}/delhi_waterscarcity_scores.csv', index=False)
with open(f'{OUTPUT_DIR}/delhi_waterscarcity_model.pkl','wb') as f:
    pickle.dump(model, f)
print(f'Saved delhi_waterscarcity_scores.csv ({len(result)} rows)')
print(result[result['year']==2023][['district','waterscarcity_risk_score','risk_category']].to_string())